# Phase 7: Gaussian-Process Validation Distributions

This notebook reports the chronological validation predictions from
the rule-specific Gaussian-process residual models.

For each decision rule, two covariance assumptions are evaluated:

\[
k_{\mathrm{RBF}}(x,x')
=
\sigma_f^2
\exp\left(
-\frac{\lVert x-x'\rVert^2}{2\ell^2}
\right),
\]

and

\[
k_{\mathrm{Mat32}}(x,x')
=
\sigma_f^2
\left(
1+\frac{\sqrt{3}\lVert x-x'\rVert}{\ell}
\right)
\exp\left(
-\frac{\sqrt{3}\lVert x-x'\rVert}{\ell}
\right).
\]

A white-noise term is added to each covariance function. Kernel
parameters are estimated using only the training rows of each fold.

The principal validation score is CRPS. This notebook does not select
the final covariance kernel and does not access market information.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve()

for candidate in (ROOT, *ROOT.parents):
    if (
        candidate
        / "data/manifests/v2/"
        "07_gp_validation_manifest.json"
    ).exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Repository root not found."
    )

manifest = json.loads(
    (
        ROOT
        / "data/manifests/v2/"
        "07_gp_validation_manifest.json"
    ).read_text(encoding="utf-8")
)

summary = pd.read_csv(
    ROOT
    / "outputs/v2/final_tables/"
    "07_gp_kernel_validation_summary.csv"
)

ledger = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "07_gp_model_fit_ledger.csv"
)

print("Status:", manifest["status"])
print()
print("Kernel validation summary:")
print(summary.to_string(index=False))
print()
print("First fitted models:")
print(
    ledger.head(8).to_string(index=False)
)

Status: TWO_YEAR_GP_VALIDATION_DISTRIBUTIONS_CERTIFIED

Kernel validation summary:
  kernel  kernel_order  validation_dates  validation_rows  fitted_models  mean_date_crps_c  median_date_crps_c  mean_row_crps_c  mean_negative_log_score  raw_mean_error_c  corrected_mean_error_c  raw_mae_c  corrected_mae_c  raw_rmse_c  corrected_rmse_c  mean_predictive_standard_deviation_c  central_50_coverage  central_80_coverage  central_90_coverage  pit_mean  pit_variance  mean_log_marginal_likelihood  total_optimizer_warnings  descriptive_crps_rank  final_kernel_selected
     rbf             1               365             1460             16          0.877561            0.660147         0.877561                 1.858892          1.448699               -0.114245   1.745685         1.253820    2.136322          1.546855                              1.436738             0.442466             0.767808             0.869178  0.480896      0.092117                   -665.573848                         0    

In [2]:
import numpy as np
import pandas as pd

predictions = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "07_gp_validation_predictions.csv"
)

paired_dates = pd.read_csv(
    ROOT
    / "outputs/v2/diagnostics/"
    "07_gp_kernel_paired_date_comparison.csv"
)

assert len(predictions) == 2920
assert predictions["target_date"].nunique() == 365
assert len(paired_dates) == 365

quantile_columns = [
    f"q{integer:02d}_c"
    for integer in range(1, 100)
]

assert np.isfinite(
    predictions[
        quantile_columns
    ].to_numpy()
).all()

assert (
    np.diff(
        predictions[
            quantile_columns
        ].to_numpy(),
        axis=1,
    )
    >= -1e-10
).all()

assert manifest["final_kernel_selected"] is False
assert manifest["market_prices_accessed"] is False

print("Mean paired CRPS difference, RBF minus Matérn 3/2:")
print(
    paired_dates[
        "rbf_minus_matern32_crps_c"
    ].mean()
)
print()
print("PHASE 7 NOTEBOOK VERIFICATION: PASSED")

Mean paired CRPS difference, RBF minus Matérn 3/2:
0.014221469481643836

PHASE 7 NOTEBOOK VERIFICATION: PASSED
